In [1]:
# Step 0: 建立项目目录并隔离
import os
from src.report.init_report_run import init_and_create_skeleton

manifest, report_md = init_and_create_skeleton(
    project_root=os.path.abspath(".."),
    run_name=None,   # 可改成 "demo_run_01"
)

print("Run dir:", manifest["paths"]["run_dir"])
print("Report :", report_md)

Run dir: D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005
Report : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\report.md


In [2]:
# Step 1 (full): 生成演示用 pop_data + 可控PK额外变异，并输出按受试者分色的PK/PD折线图
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data.simulate_pkpd import generate_population_data

if "manifest" not in globals():
    raise RuntimeError("请先执行 Step 0，确保 manifest 已初始化。")

# =========================
# 0) 参数
# =========================
model_name = "IDR_INHIB_KIN_SIG"
seed = 42
n_subjects = 24
extra_pk_iiv_sigma = 0.10

# 统一使用当前run目录
run_dir = manifest["paths"]["run_dir"]
data_dir = os.path.join(run_dir, "data")
os.makedirs(data_dir, exist_ok=True)

# =========================
# 1) 调用src生成数据
# =========================
ret = generate_population_data(
    model_name=model_name,
    seed=seed,
    n_subjects=n_subjects,
    extra_pk_iiv_sigma=extra_pk_iiv_sigma,
    return_pk_scale=True,
)

pop_data, subject_params, cfg, pk_scale_by_sid = ret

df = pd.DataFrame(pop_data, columns=["sid", "time", "C_obs", "R_obs"])
df["sid"] = df["sid"].astype(int)

# =========================
# 2) 保存到 run_dir/data
# =========================
csv_path = os.path.join(data_dir, "pkpd_long.csv")
cfg_path = os.path.join(data_dir, "sim_config.json")
subj_path = os.path.join(data_dir, "subject_params.json")
fig_pkpd_lines = os.path.join(data_dir, "pkpd_lines_by_subject.png")

df.to_csv(csv_path, index=False)

def _jsonable(x):
    if isinstance(x, dict):
        return {k: _jsonable(v) for k, v in x.items()}
    if isinstance(x, list):
        return [_jsonable(v) for v in x]
    if isinstance(x, tuple):
        return tuple(_jsonable(v) for v in x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    return x

meta = {
    "data_source": "simulated_from_src",
    "generator": "src.data.simulate_pkpd.generate_population_data",
    "model_name": model_name,
    "seed": seed,
    "n_subjects": n_subjects,
    "extra_pk_iiv_sigma": float(extra_pk_iiv_sigma),
    "pk_scale_by_sid": _jsonable(pk_scale_by_sid) if pk_scale_by_sid is not None else None,
    "cfg": _jsonable(cfg),
    "n_rows": int(len(df)),
    "n_time_unique": int(df["time"].nunique()),
    "run_dir": run_dir,
}

with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

with open(subj_path, "w", encoding="utf-8") as f:
    json.dump(_jsonable(subject_params), f, ensure_ascii=False, indent=2)

# =========================
# 3) 按受试者分色折线图（PK/PD）
# =========================
df_plot = df.sort_values(["sid", "time"]).copy()
sids = sorted(df_plot["sid"].unique())
cmap = plt.get_cmap("tab20", len(sids))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharex=True)

for i, sid in enumerate(sids):
    g = df_plot[df_plot["sid"] == sid]
    axes[0].plot(g["time"], g["C_obs"], color=cmap(i), lw=1.4, alpha=0.9)
axes[0].set_title("PK profiles by subject (C_obs vs time)")
axes[0].set_xlabel("time")
axes[0].set_ylabel("C_obs")
axes[0].grid(alpha=0.2)

for i, sid in enumerate(sids):
    g = df_plot[df_plot["sid"] == sid]
    axes[1].plot(g["time"], g["R_obs"], color=cmap(i), lw=1.4, alpha=0.9)
axes[1].set_title("PD profiles by subject (R_obs vs time)")
axes[1].set_xlabel("time")
axes[1].set_ylabel("R_obs")
axes[1].grid(alpha=0.2)

handles = [plt.Line2D([0], [0], color=cmap(i), lw=2) for i in range(len(sids))]
labels = [f"sid={sid}" for sid in sids]
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8, frameon=False)

plt.tight_layout()
plt.savefig(fig_pkpd_lines, dpi=160, bbox_inches="tight")
plt.close()

print("Data generated from src successfully.")
print("RUN      :", run_dir)
print("DATA DIR :", data_dir)
print("CSV      :", csv_path)
print("CFG      :", cfg_path)
print("SUBJ     :", subj_path)
print("FIG      :", fig_pkpd_lines)
print("shape    :", pop_data.shape)
df.head()

Data generated from src successfully.
RUN      : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005
DATA DIR : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\data
CSV      : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\data\pkpd_long.csv
CFG      : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\data\sim_config.json
SUBJ     : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\data\subject_params.json
FIG      : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\data\pkpd_lines_by_subject.png
shape    : (288, 4)


,sid,time,C_obs,R_obs
0,0,0.0,0.000000,34.889863
1,0,0.5,3.673347,32.437015
2,0,1.0,5.496231,26.811411
3,0,2.0,6.521732,16.771803
4,0,3.0,6.126612,11.915866


In [3]:
# Step 2: 使用src函数做 summary + plot；报告写入（按模板真实标题替换）
import os, re, json
import numpy as np
import pandas as pd

from src.report.summary_plot import (
    load_pkpd_csv,
    load_optional_json,
    save_snapshot_and_summary,
    plot_pkpd_scatter,
)

if "manifest" not in globals():
    raise RuntimeError("请先执行 Step 0，确保 manifest 已初始化。")

run_dir = manifest["paths"]["run_dir"]
fig_dir = manifest["paths"]["figures_dir"]
tables_dir = manifest["paths"]["tables_dir"]
report_md = manifest["paths"]["report_md"]
manifest_json = manifest["paths"]["manifest_json"]

csv_path = os.path.join(run_dir, "data", "pkpd_long.csv")
cfg_path = os.path.join(run_dir, "data", "sim_config.json")

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"未找到数据文件: {csv_path}")

df = load_pkpd_csv(csv_path)
cfg = load_optional_json(cfg_path) if os.path.exists(cfg_path) else {}
snap_csv, summary_json, summary = save_snapshot_and_summary(df, tables_dir)
fig_path = plot_pkpd_scatter(df, fig_dir, filename="pkpd_scatter.png")

# ---- 自动从数据时间分布生成 VPC 分箱边界 ----
unique_times = sorted(df["time"].dropna().unique())
if len(unique_times) <= 6:
    bin_edges = unique_times
else:
    percentiles = np.linspace(0, 100, 6)
    bin_edges = np.percentile(unique_times, percentiles).tolist()
    bin_edges[0] = float(df["time"].min())
    bin_edges[-1] = float(df["time"].max())
bin_edges = sorted(set(bin_edges))

manifest["config"]["bin_edges"] = bin_edges
with open(manifest_json, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print(f"bin_edges auto-computed: {bin_edges}")

goal_text = "本报告展示一个可复现的 PK/PD 全流程：从数据可视化、候选结构发现、多起点参数精调、模型筛选，到诊断图与方程复原。"
pk_text = cfg.get("pk_model_text", "PK 模型由输入数据给定（可来自模拟或外部观测）。")
pd_text = cfg.get("pd_model_text", "PD 模型由结构发现与参数拟合共同确定。")

tv = cfg.get("typical_pd_params", {})
if not tv and isinstance(cfg.get("cfg"), dict):
    tv = cfg["cfg"].get("tv_pd", {})

data_text = (
    f"- 数据来源: **{cfg.get('data_source', 'unknown')}**\n"
    f"- 样本点数: **{summary['n_rows']}**\n"
    f"- 受试者数: **{summary['n_subjects']}**\n"
    f"- time 范围: **[{summary['time_min']:.3f}, {summary['time_max']:.3f}]**\n"
    f"- C_obs 范围: **[{summary['C_min']:.3f}, {summary['C_max']:.3f}]**\n"
    f"- R_obs 范围: **[{summary['R_min']:.3f}, {summary['R_max']:.3f}]**\n"
    f"- VPC bin_edges: **{bin_edges}**"
)
if tv:
    data_text += "\n- Typical PD params (TV): " + ", ".join([f"{k}={v}" for k, v in tv.items()])

fig_text = "![PKPD Scatter](figures/pkpd_scatter.png)"

def replace_section(md_text, section_title, new_body):
    pattern = rf"({re.escape(section_title)}\n)(.*?)(?=\n### |\n## |\Z)"
    repl = rf"\1{new_body}\n"
    if re.search(pattern, md_text, flags=re.S):
        return re.sub(pattern, repl, md_text, flags=re.S)
    return md_text + f"\n\n{section_title}\n{new_body}\n"

with open(report_md, "r", encoding="utf-8") as f:
    md = f.read()

md = replace_section(md, "### 1.1 研究目标", goal_text)
md = replace_section(md, "### 1.2 已知 PK 方程", pk_text)
md = replace_section(md, "### 1.3 已知 PD 方程", pd_text)
md = replace_section(md, "### 2.1 数据来源与统计", data_text)
md = replace_section(md, "### 2.2 PK/PD 散点图", fig_text)

with open(report_md, "w", encoding="utf-8") as f:
    f.write(md)

print("Step 2 done.")
print("figure  :", fig_path)
print("snapshot:", snap_csv)
print("summary :", summary_json)
print("report  :", report_md)

bin_edges auto-computed: [0.0, 1.2000000000000002, 3.4000000000000004, 7.199999999999999, 11.600000000000001, 24.0]
Step 2 done.
figure  : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\figures\pkpd_scatter.png
snapshot: D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\tables\pkpd_long_snapshot.csv
summary : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\tables\data_summary.json
report  : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\report.md


In [4]:
# Step 3 (final, simplified): 结构发现 + 候选导出（Step 4 直接复用 tables/topk_candidates.json）
import os
import re
import json
import pandas as pd

from src.pipeline.discovery import run_single_discovery
from src.configs.defaults import DEFAULTS

# ========= 0) 路径 =========
run_dir = manifest["paths"]["run_dir"]
tables_dir = manifest["paths"]["tables_dir"]
report_md = manifest["paths"]["report_md"]

# ========= 1) 读取数据 =========
csv_path = os.path.join(run_dir, "data", "pkpd_long.csv")
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"未找到数据文件: {csv_path}")

df = pd.read_csv(csv_path)
required_cols = {"sid", "time", "C_obs", "R_obs"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"CSV缺少必要列，当前列: {list(df.columns)}")

df["sid"] = df["sid"].astype(int)
pop_data = df[["sid", "time", "C_obs", "R_obs"]].values

# ========= 2) 结构发现 =========
config = dict(DEFAULTS)
active_model = "IDR_INHIB_KIN_SIG"

print("[Step 3] Running discovery...")
res = run_single_discovery(
    pop_data=pop_data,
    active_model=active_model,
    config=config,
)

if not isinstance(res, dict) or "top_results" not in res or len(res["top_results"]) == 0:
    raise RuntimeError("结构发现结果为空或格式异常。")

top_results = res["top_results"]

# ========= 3) 生成 topk_candidates.json（Step 4 直接复用，不需要 export_nlme_inputs） =========
candidate_json = os.path.join(tables_dir, "topk_candidates.json")

# 构造标准 JSON（与 export_nlme_inputs 格式完全一致，Step 4 直接读）
std_candidates = []
for i, c in enumerate(top_results, 1):
    terms = c.get("terms", [])
    score_bic_val = c.get("score", c.get("score_bic_val"))
    mse_val = c.get("mse_val")
    mse_train = c.get("mse_train")

    std_candidates.append({
        "candidate_id": i,
        "rank": i,
        "active_model": active_model,
        "terms": terms,
        "k": int(c.get("k", len(terms))),
        "score_bic_val": float(score_bic_val) if score_bic_val is not None else float("nan"),
        "mse_val": float(mse_val) if mse_val is not None else float("nan"),
        "mse_train": float(mse_train) if mse_train is not None else float("nan"),
        "init_params": [],
        # Step 3 只写全局 ec50/gamma（Step 4 会用拟合值覆盖）
        "ec50_hat": float(res.get("ec50_hat", 4.0) or 4.0),
        "gamma_hat": float(res.get("gamma_hat", 2.0) or 2.0),
    })

payload = {
    "active_model": active_model,
    "ec50_hat": float(res.get("ec50_hat", 4.0) or 4.0),
    "gamma_hat": float(res.get("gamma_hat", 2.0) or 2.0),
    "candidates": std_candidates,
}

with open(candidate_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

# 写标准化 CSV
candidate_csv = os.path.join(tables_dir, "topk_candidates.csv")
rows_std = []
for c in std_candidates:
    rows_std.append({
        "candidate_id": c["candidate_id"],
        "rank": c["rank"],
        "active_model": c["active_model"],
        "selected_terms": " + ".join([str(x) for x in c["terms"]]),
        "k": c["k"],
        "score_bic_val": c["score_bic_val"],
        "mse_val": c["mse_val"],
        "mse_train": c["mse_train"],
        "init_params_json": json.dumps(c["init_params"], ensure_ascii=False),
        "ec50_hat": c["ec50_hat"],
        "gamma_hat": c["gamma_hat"],
    })

pd.DataFrame(rows_std).sort_values(["rank", "candidate_id"]).to_csv(candidate_csv, index=False)

# ========= 4) 回填报告第3章（按模板标题替换，防错位） =========
def to_md_table(cands, max_rows=50):
    cands = cands[:max_rows]
    lines = [
        "| rank | active_model | selected_terms | k | score_bic_val |",
        "|---:|---|---|---:|---:|",
    ]
    for c in cands:
        terms = " + ".join([str(x) for x in c.get("terms", [])])
        val = c.get("score_bic_val", "")
        lines.append(f"| {c.get('rank','')} | {c.get('active_model','')} | {terms} | {c.get('k','')} | {val} |")
    return "\n".join(lines)

sec31 = (
    "候选项库示例：`1, R, C, C^2, Emax(C), Hill(C), C*R, Emax(C)*R, Hill(C)*R`。\n\n"
    f"本次结构发现采用 `active_model={active_model}`，`topk={config.get('topk', 'NA')}`。"
)

sec32 = (
    "Top-K 候选结构如下（由 `export_nlme_inputs` 导出并标准化）：\n\n"
    + to_md_table(std_candidates, max_rows=50)
    + "\n\n详细文件：`tables/topk_candidates.csv`、`tables/topk_candidates.json`"
)

def replace_section(md_text, section_title, new_body):
    pattern = rf"({re.escape(section_title)}\n)(.*?)(?=\n### |\n## |\Z)"
    repl = rf"\1{new_body}\n"
    if re.search(pattern, md_text, flags=re.S):
        return re.sub(pattern, repl, md_text, flags=re.S)
    return md_text + f"\n\n{section_title}\n{new_body}\n"

with open(report_md, "r", encoding="utf-8") as f:
    md = f.read()

# 注意：模板是三级标题
md = replace_section(md, "### 3.1 候选项库", sec31)
md = replace_section(md, "### 3.2 剪枝与 Top-K 结构", sec32)

with open(report_md, "w", encoding="utf-8") as f:
    f.write(md)

print("\nStep 3 completed (simplified, fixed field mapping).")
print("TopK JSON:", candidate_json)
print("TopK CSV :", candidate_csv)
print("Report   :", report_md)
print("Top-1    :", std_candidates[0] if std_candidates else None)

[Step 3] Running discovery...
Preprocessing data
Dataset is using device:  cuda

[Warmup]
warmup done | loss=0.006544 mse=0.006260 reg=0.000371

[Iterative pruning]
Round 1:
  active(old): [np.str_('1'), np.str_('R'), np.str_('C'), np.str_('C^2'), np.str_('Emax(C)'), np.str_('Hill(C)'), np.str_('C*R'), np.str_('Emax(C)*R'), np.str_('Hill(C)*R')]
  active(new): [np.str_('1'), np.str_('R'), np.str_('Hill(C)')]
  retrain done | loss=0.000328 mse=0.000113 reg=0.000017
Round 2:
  active(old): [np.str_('1'), np.str_('R'), np.str_('Hill(C)')]
  active(new): [np.str_('1'), np.str_('R'), np.str_('Hill(C)')]
  -> unchanged (1/2)
Round 3:
  active(old): [np.str_('1'), np.str_('R'), np.str_('Hill(C)')]
  active(new): [np.str_('1'), np.str_('R'), np.str_('Hill(C)')]
  -> unchanged (2/2)
  -> mask stable enough, stop.

[Top-K candidate generation with Validation-BIC]

=== Top-K Candidate Structures (Validation BIC) ===
[Rank 1] BIC_val=-0.418120 | mse_val=0.318462 | mse_train=0.000052 | k=3
  terms:

In [5]:
# Step 4 (prep-for-NLME): 多起点轻量精调（用于结构与初值准备，不做最终参数结论）
import os, re, json, pandas as pd, matplotlib.pyplot as plt
from src.pipeline.multistart_refit import run_multistart_refit

if "manifest" not in globals():
    raise RuntimeError("请先执行 Step 0，确保 manifest 已初始化。")

run_dir = manifest["paths"]["run_dir"]
tables_dir = manifest["paths"]["tables_dir"]
fig_dir = manifest["paths"]["figures_dir"]
report_md = manifest["paths"]["report_md"]

csv_path = os.path.join(run_dir, "data", "pkpd_long.csv")
topk_json = os.path.join(tables_dir, "topk_candidates.json")

if not os.path.exists(csv_path):
    raise RuntimeError(f"未找到数据文件: {csv_path}")
if not os.path.exists(topk_json):
    raise RuntimeError(f"未找到候选文件: {topk_json}")

df = pd.read_csv(csv_path)
with open(topk_json, "r", encoding="utf-8") as f:
    topk_payload = json.load(f)

df_all, df_best = run_multistart_refit(
    df_long=df,
    topk_payload=topk_payload,
    n_restarts=8,
    seed=20260419,
    theta_scale=0.3,
    max_nfev=1200,
    train_frac=0.7,
    keep_time_order=True,
    use_bounds=True,
    theta_abs_bound=50.0,   # 改为50.0（原10.0会截断真值 Kin=25, -Kin*Imax=-25）
    lambda_k=0.0,
)

all_csv = os.path.join(tables_dir, "multistart_all_runs.csv")
best_csv = os.path.join(tables_dir, "multistart_summary.csv")
df_all.to_csv(all_csv, index=False)
df_best.to_csv(best_csv, index=False)

# ---- 把 Step 4 计算出的 per-candidate 参数全部写回 topk_candidates.json ----
# 包含: ec50_hat, gamma_hat, theta_hat（供 MATLAB 拟合器使用）
with open(topk_json, "r", encoding="utf-8") as f:
    payload = json.load(f)

for c in payload.get("candidates", []):
    cid = int(c.get("candidate_id", c.get("rank", 0)))
    row = df_best[df_best["candidate_id"] == cid]
    if row.empty:
        continue
    r = row.iloc[0]

    # --- ec50 / gamma ---
    c["ec50_hat"] = float(r["ec50_hat"]) if pd.notna(r.get("ec50_hat")) else 4.0
    c["gamma_hat"] = float(r["gamma_hat"]) if pd.notna(r.get("gamma_hat")) else 2.0

    # --- theta_hat：结构参数初值（前 p 个，p = k - has_hill*2）---
    terms = c.get("terms", [])
    has_hill = any(t in ("Hill(C)", "Hill(C)*R") for t in terms)
    p = len(terms)  # 结构参数数量

    theta_full = []
    try:
        theta_full = json.loads(r.get("theta_hat_json", "[]"))
    except Exception:
        theta_full = []

    if theta_full and len(theta_full) >= p:
        # theta_full = [theta1..theta_p, ec50, gamma]（如果 has_hill）
        c["theta_hat"] = [float(v) for v in theta_full[:p]]
    else:
        c["theta_hat"] = []

with open(topk_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

# ---- BIC 分布图 ----
plt.figure(figsize=(10, 5))
for cid, g in df_all.groupby("candidate_id"):
    if "BIC_val" not in g.columns:
        continue
    y = g["BIC_val"].replace([float("inf"), float("-inf")], pd.NA).dropna().values
    if len(y) == 0:
        continue
    plt.scatter([cid] * len(y), y, s=18, alpha=0.7)
plt.xlabel("candidate_id")
plt.ylabel("BIC_val across restarts")
plt.title("Multi-start BIC_val distribution (for NLME initialization prep)")
plt.grid(alpha=0.2)
fig_path = os.path.join(fig_dir, "multistart_bic_scatter.png")
plt.tight_layout()
plt.savefig(fig_path, dpi=160, bbox_inches="tight")
plt.close()

# ---- 回填报告 ----
top1 = df_best.iloc[0].to_dict() if len(df_best) else {}

sec41 = (
    "本步骤用于 **NLME 前准备**：在候选结构内进行多起点拟合，输出可复用的结构排序与参数初值。\n\n"
    "- 输入：`tables/topk_candidates.json`\n"
    "- 全部重启结果：`tables/multistart_all_runs.csv`\n"
    "- 各候选最佳重启：`tables/multistart_summary.csv`\n"
    "- 参考图：`figures/multistart_bic_scatter.png`\n\n"
    "说明：本步骤不作为最终参数估计结论，最终比较在 NLME 阶段完成。"
)

sec42 = (
    f"当前首位候选（仅作初始化参考）：candidate_id={top1.get('candidate_id', 'NA')}, "
    f"terms=`{top1.get('terms', 'NA')}`, "
    f"BIC_val={top1.get('BIC_val', 'NA')}, "
    f"AIC_val={top1.get('AIC_val', 'NA')}。\n\n"
    "![Multi-start BIC](figures/multistart_bic_scatter.png)"
)

# ---- sec43: 各候选参数表 ----
def make_sec43(df_best):
    lines = [
        "| candidate_id | rank | terms | k | BIC_val | theta_hat |",
        "|---:|---:|---|---|---:|---|",
    ]
    for _, row in df_best.iterrows():
        terms = row.get("terms", "NA")
        try:
            th = json.loads(row.get("theta_hat_json", "[]"))
            th_str = ", ".join([f"{v:.4f}" for v in th])
        except Exception:
            th_str = "NA"
        lines.append(
            f"| {int(row['candidate_id'])} | {int(row['rank'])} | "
            f"`{terms}` | {int(row['k'])} | {row.get('BIC_val', 'NA'):.4f} | [{th_str}] |"
        )
    return "\n".join(lines)

sec43 = (
    "各候选最佳重启参数（归一化尺度，仅供 NLME 初始化参考）：\n\n"
    + make_sec43(df_best)
    + "\n\n详细文件：`tables/multistart_summary.csv`"
)

def replace_section(md_text, section_title, new_body):
    pattern = rf"({re.escape(section_title)}\n)(.*?)(?=\n### |\n## |\Z)"
    repl = rf"\1{new_body}\n"
    if re.search(pattern, md_text, flags=re.S):
        return re.sub(pattern, repl, md_text, flags=re.S)
    return md_text + f"\n\n{section_title}\n{new_body}\n"

with open(report_md, "r", encoding="utf-8") as f:
    md = f.read()

md = replace_section(md, "### 4.1 多起点配置", sec41)
md = replace_section(md, "### 4.2 各候选最优参数与初值", sec42)
md = replace_section(md, "### 4.3 各候选参数表", sec43)

with open(report_md, "w", encoding="utf-8") as f:
    f.write(md)

print("Step 4 done (prep-for-NLME).")
print("all runs :", all_csv)
print("summary  :", best_csv)
print("figure   :", fig_path)
print("JSON     :", topk_json)

Step 4 done (prep-for-NLME).
all runs : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\tables\multistart_all_runs.csv
summary  : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\tables\multistart_summary.csv
figure   : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\figures\multistart_bic_scatter.png
JSON     : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\tables\topk_candidates.json


In [6]:
# Step 5 (final): NLME群体验证与模型比较（复用src + MATLAB SimBiology）
import os, re, json, numpy as np, pandas as pd
from src.pipeline.run_simbiology_validation import run_simbiology_validation

if "manifest" not in globals():
    raise RuntimeError("请先执行 Step 0，确保 manifest 已初始化。")

run_dir = manifest["paths"]["run_dir"]
tables_dir = manifest["paths"]["tables_dir"]
report_md = manifest["paths"]["report_md"]

csv_path = os.path.join(run_dir, "data", "pkpd_long.csv")
topk_json = os.path.join(tables_dir, "topk_candidates.json")
ms_summary_csv = os.path.join(tables_dir, "multistart_summary.csv")

nlme_csv = os.path.join(tables_dir, "simbiology_results.csv")
model_cmp_csv = os.path.join(tables_dir, "model_compare.csv")

df = pd.read_csv(csv_path)
required_cols = {"sid", "time", "C_obs", "R_obs"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"CSV缺少必要列，当前列: {list(df.columns)}")
df["sid"] = df["sid"].astype(int)
pop_data = df[["sid", "time", "C_obs", "R_obs"]].values

with open(topk_json, "r", encoding="utf-8") as f:
    payload = json.load(f)

top_results = payload.get("candidates", [])
if len(top_results) == 0:
    raise RuntimeError("topk_candidates.json 的 candidates 为空，无法进行NLME验证。")

if os.path.exists(ms_summary_csv):
    df_ms = pd.read_csv(ms_summary_csv)
    if "candidate_id" in df_ms.columns:
        order = [int(x) for x in df_ms["candidate_id"].dropna().tolist()]
        rank_map = {cid: i for i, cid in enumerate(order)}
        top_results = sorted(
            top_results,
            key=lambda c: rank_map.get(int(c.get("candidate_id", c.get("rank", 10**9))), 10**9)
        )

# use_population_mean=True：与 Step 4 一致，用群体均值数据拟合
df_nlme = run_simbiology_validation(
    pop_data=pop_data,
    top_results=top_results,
    project_root="..",
    out_dir=tables_dir,
    use_population_mean=True,
)

df_nlme.to_csv(nlme_csv, index=False)

# ---- 构建模型比较表 ----
if "candidate_id" in df_nlme.columns:
    df_cmp = df_nlme.copy()
    df_cmp["candidate_id"] = pd.to_numeric(df_cmp["candidate_id"], errors="coerce").astype("Int64")
elif "rank" in df_nlme.columns:
    df_cmp = df_nlme.copy()
    df_cmp["candidate_id"] = pd.to_numeric(df_cmp["rank"], errors="coerce").astype("Int64")
else:
    df_cmp = df_nlme.copy()
    df_cmp["candidate_id"] = pd.Series(np.arange(1, len(df_cmp) + 1), dtype="Int64")

# ---- 直接从 top_results 建立 terms->k 映射 ----
terms_k_map = {}
for c in top_results:
    terms_list = c.get("terms", [])
    k_val = int(c.get("k", len(terms_list))) if c.get("k") is not None else len(terms_list)
    terms_key = " + ".join([str(x) for x in terms_list])
    terms_k_map[terms_key] = k_val

df_cmp["terms"] = df_cmp["terms"]
df_cmp["k"] = df_cmp["terms"].map(
    lambda x: terms_k_map.get(x, len(x.split(" + ")) if pd.notna(x) else np.nan)
)

metric_candidates = ["BIC", "bic", "BIC_val", "AIC", "aic", "AIC_val", "OFV", "ofv", "NLL", "nll", "OBJ", "obj"]
metric_col = None
for c in metric_candidates:
    if c in df_cmp.columns:
        s = pd.to_numeric(df_cmp[c], errors="coerce")
        if s.notna().any():
            metric_col = c
            break
if metric_col is None:
    raise ValueError(f"NLME结果中未找到可排序指标列。当前列: {list(df_cmp.columns)}")

df_cmp[metric_col] = pd.to_numeric(df_cmp[metric_col], errors="coerce")
df_cmp = df_cmp.sort_values(metric_col, ascending=True, na_position="last").reset_index(drop=True)
best_metric = df_cmp[metric_col].dropna().iloc[0] if df_cmp[metric_col].notna().any() else np.nan
df_cmp["delta_metric"] = df_cmp[metric_col] - best_metric if pd.notna(best_metric) else np.nan
df_cmp["model_rank"] = np.arange(1, len(df_cmp) + 1)

preferred_cols = [
    "model_rank", "candidate_id", "terms", "k",
    metric_col, "delta_metric",
    "AIC", "aic", "AIC_val",
    "BIC", "bic", "BIC_val",
    "OFV", "ofv", "NLL", "nll",
    "converged", "status", "message"
]
seen, out_cols = set(), []
for c in preferred_cols:
    if c in df_cmp.columns and c not in seen:
        out_cols.append(c)
        seen.add(c)
for c in ["model_rank", "candidate_id", "terms", "k", metric_col, "delta_metric"]:
    if c not in out_cols:
        out_cols.append(c)

df_out = df_cmp[out_cols].copy()
df_out.to_csv(model_cmp_csv, index=False)

# ---- 回填报告 ----
def to_md_table(df_in, max_rows=20):
    show = df_in.head(max_rows).copy()
    for col in show.columns:
        if pd.api.types.is_numeric_dtype(show[col]):
            show[col] = show[col].map(lambda x: f"{x:.4f}" if pd.notna(x) else "")
    try:
        return show.to_markdown(index=False)
    except Exception:
        return show.to_string(index=False)

top1 = df_out.iloc[0].to_dict() if len(df_out) else {}

sec51 = (
    f"基于 MATLAB SimBiology/NLME 的群体层比较结果如下，排序指标为 `{metric_col}`（越小越优）：\n\n"
    + to_md_table(df_out, max_rows=50)
    + "\n\n详细文件：`tables/simbiology_results.csv`、`tables/model_compare.csv`"
)

sec52 = (
    f"最终模型排序以 NLME 指标 `{metric_col}` 为准（Step4 仅用于初始化筛选）。\n\n"
    f"当前最优模型：\n"
    f"- rank={int(top1.get('model_rank', 1)) if len(df_out) else 'NA'}\n"
    f"- candidate_id={top1.get('candidate_id', 'NA')}\n"
    f"- terms=`{top1.get('terms', 'NA')}`\n"
    f"- k={int(top1.get('k')) if pd.notna(top1.get('k')) else 'NA'}\n"
    f"- {metric_col}={top1.get(metric_col, 'NA')}\n"
    f"- delta_metric={top1.get('delta_metric', 'NA')}\n\n"
    f"结论：该候选在群体参数层面的 NLME 比较中排名第一。"
)

# ---- sec53: NLME 拟合参数表 ----
def make_sec53(df_nlme_in, top_results_ref):
    k_map = {}
    for c in top_results_ref:
        tk = " + ".join([str(x) for x in c.get("terms", [])])
        k_val = int(c.get("k")) if c.get("k") is not None else len(c.get("terms", []))
        k_map[tk] = k_val
    theta_cols = [col for col in df_nlme_in.columns if col.startswith("theta")]
    n_th = len(theta_cols)
    # header: 4 fixed cols (rank, terms, k, + blank before theta1) + n_th theta cols = 4 + n_th
    # Header row: "| rank | terms | k | theta1 | theta2 | ... |"
    th_header = " | ".join([f"theta{i}" for i in range(1, n_th + 1)])
    header_row = f"| rank | terms | k | {th_header} |"
    # Separator: 4 cells for first 4 cols, then 1 per theta = 4 + n_th
    sep = "|---:|---|---|" + "|---:" * n_th + "|"
    lines = [header_row, sep]
    for _, row in df_nlme_in.iterrows():
        terms_str = str(row.get("terms", "NA"))
        k = k_map.get(terms_str, len(terms_str.split(" + ")) if pd.notna(row.get("terms")) else n_th)
        th_vals = []
        for i in range(1, n_th + 1):
            col = f"theta{i}"
            if col in row.index and pd.notna(row[col]):
                th_vals.append(f"{row[col]:.4f}")
            else:
                th_vals.append("NA")
        lines.append(
            f"| {int(row['rank'])} | `{terms_str}` | {k} | " + " | ".join(th_vals) + " |"
        )
    return "\n".join(lines)

sec53 = (
    "NLME 拟合参数（原始物理尺度）：\n\n"
    + make_sec53(df_nlme, top_results)
    + "\n\n详细文件：`tables/simbiology_results.csv`"
)

def replace_section(md_text, section_title, new_body):
    pattern = rf"({re.escape(section_title)}\n)(.*?)(?=\n### |\n## |\Z)"
    repl = rf"\1{new_body}\n"
    if re.search(pattern, md_text, flags=re.S):
        return re.sub(pattern, repl, md_text, flags=re.S)
    return md_text + f"\n\n{section_title}\n{new_body}\n"

with open(report_md, "r", encoding="utf-8") as f:
    md = f.read()

md = replace_section(md, "### 5.1 拟合优度指标", sec51)
md = replace_section(md, "### 5.2 模型排序结论", sec52)
md = replace_section(md, "### 5.3 NLME 参数估计", sec53)

with open(report_md, "w", encoding="utf-8") as f:
    f.write(md)

print("Step 5 done (NLME final).")
print("NLME raw :", nlme_csv)
print("Compare  :", model_cmp_csv)
print("Metric   :", metric_col)
print("Top1     :", top1)

Step 5 done (NLME final).
NLME raw : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\tables\simbiology_results.csv
Compare  : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\tables\model_compare.csv
Metric   : BIC
Top1     : {'model_rank': 1, 'candidate_id': 1, 'terms': '1 + R + Emax(C) + Hill(C)', 'k': 4, 'BIC': -9.77945885392714, 'delta_metric': 0.0, 'AIC': -12.6888987526551, 'converged': 1, 'message': nan}


In [7]:
# Step 6: 诊断图 - GOF / 残差 / VPC / Bootstrap（调用 MATLAB 诊断脚本）
import os, json, pandas as pd

if "manifest" not in globals():
    raise RuntimeError("请先执行 Step 0，确保 manifest 已初始化。")

run_dir = manifest["paths"]["run_dir"]
fig_dir = manifest["paths"]["figures_dir"]
tables_dir = manifest["paths"]["tables_dir"]
report_md = manifest["paths"]["report_md"]
manifest_json = manifest["paths"]["manifest_json"]

with open(manifest_json, "r", encoding="utf-8") as f:
    meta = json.load(f)
bin_edges = meta.get("config", {}).get("bin_edges", [0.0, 1.0, 4.0, 8.0, 24.0])
print(f"Using bin_edges from manifest: {bin_edges}")

csv_path = os.path.join(run_dir, "data", "pkpd_long.csv")
model_cmp_csv = os.path.join(tables_dir, "model_compare.csv")
simbio_csv = os.path.join(tables_dir, "simbiology_results.csv")
topk_json = os.path.join(tables_dir, "topk_candidates.json")

df_cmp = pd.read_csv(model_cmp_csv)
n_candidates = len(df_cmp)

# ---- 调用 MATLAB 诊断脚本 ----
from src.pipeline.run_simbiology_validation import run_simbiology_diagnostics

print("Running MATLAB diagnostics (VPC only, skip Bootstrap)...")
run_simbiology_diagnostics(
    data_csv=csv_path,
    simbio_csv=simbio_csv,
    topk_json=topk_json,
    fig_dir=fig_dir,
    project_root="..",
    bin_edges=bin_edges,
    skip_bootstrap=True,   # Bootstrap 很耗时，先跳过看其他结果
)
print("MATLAB diagnostics done.")

# ---- 回填报告（按标题精确替换，不越界）----
df_nlme = pd.read_csv(simbio_csv)

# sec61: 模型汇总表（含 BIC/AIC）
tbl61 = ["| Rank | Terms | k | BIC | AIC | converged |",
         "|---:|---|---|---:|---:|---:|"]
for _, row in df_nlme.iterrows():
    k_val = int(row.get("k", len(str(row.get("terms", "")).split(" + "))))
    tbl61.append(
        f"| {int(row['rank'])} | `{row['terms']}` | {k_val} | "
        f"{row['BIC']:.4f} | {row['AIC']:.4f} | {'Y' if row.get('converged', 0) else 'N'} |"
    )
sec61_body = (
    f"NLME 候选共 {n_candidates} 个，诊断结果汇总如下：\n\n"
    + "\n".join(tbl61)
    + f"\n\n![Step6 Overview](figures/step6_overview.png)"
)

# sec62: GOF + 残差
sec62_body = "\n\n---\n\n".join([
    f"### Rank {r}\n\n![m{r} GOF](figures/m{r}_gof.png)\n\n![m{r} Residual](figures/m{r}_residual.png)"
    for r in range(1, n_candidates + 1)
])

# sec63: Bootstrap（跳过时提示）
sec63_body = "\n\n---\n\n".join([
    f"### Rank {r} Bootstrap\n\n*Bootstrap skipped in this run.*"
    for r in range(1, n_candidates + 1)
])

# sec64: VPC
sec64_body = "\n\n---\n\n".join([
    f"### Rank {r} VPC\n\n![m{r} VPC](figures/m{r}_vpc.png)"
    for r in range(1, n_candidates + 1)
])

def replace_section(md_text, section_title, new_body):
    """
    精确替换：以 section_title 开头、下一同级标题或文档末尾截止。
    不使用非贪婪匹配，避免跨越多个同级标题。
    """
    # 找 section_title 所在行
    lines = md_text.split("\n")
    title_line_idx = None
    for i, line in enumerate(lines):
        if line.strip() == section_title.strip():
            title_line_idx = i
            break
    if title_line_idx is None:
        return md_text

    # 找下一个 ## 或 ### 标题（同级或更高级）
    next_idx = None
    for i in range(title_line_idx + 1, len(lines)):
        stripped = lines[i].strip()
        if stripped.startswith("## ") or stripped.startswith("### "):
            next_idx = i
            break

    # 重建
    before = "\n".join(lines[:title_line_idx])
    if next_idx is not None:
        after = "\n".join(lines[next_idx:])
    else:
        after = ""

    return before + "\n" + section_title + "\n" + new_body + "\n" + after

with open(report_md, "r", encoding="utf-8") as f:
    md = f.read()

md = replace_section(md, "### 6.1 模型汇总", sec61_body)
md = replace_section(md, "### 6.2 GOF 与残差", sec62_body)
md = replace_section(md, "### 6.3 Bootstrap", sec63_body)
md = replace_section(md, "### 6.4 VPC", sec64_body)

with open(report_md, "w", encoding="utf-8") as f:
    f.write(md)

print(f"\nStep 6 done.")
print(f"Overview: {os.path.join(fig_dir, 'step6_overview.png')}")
print(f"Report :", report_md)

Using bin_edges from manifest: [0.0, 1.2000000000000002, 3.4000000000000004, 7.199999999999999, 11.600000000000001, 24.0]
Running MATLAB diagnostics (VPC only, skip Bootstrap)...
MATLAB diagnostics done.

Step 6 done.
Overview: D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\figures\step6_overview.png
Report : D:\PycharmProjects\DeePyMoD\reports\run_20260429_150005\report.md
